# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [179]:
class Matrix:
    def __init__(self, n, m=None):

        if isinstance(n, int) and isinstance(m, int):
            self.rows = n
            self.cols = m
            self.data = [[0.0 for _ in range(m)] for _ in range(n)]

        elif isinstance(n, list):
            if len(n) == 0:
                raise ValueError("Matrix cannot be empty")

            row_length = len(n[0])

            for row in n:
                if len(row) != row_length:
                    raise ValueError("All rows must have the same number of columns")
                    
            self.rows = len(n)
            self.cols = row_length
            self.data = [[float(value) for value in row] for row in n]
            
        else:
            raise TypeError("Invalid constructor usage")

    def shape(self):
        return (self.rows, self.cols)
        
    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]
            
        return self.data[key]
        
    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = float(value)

        else:
            if len(value) != self.cols:
                raise ValueError("Row must have correct number of columns")
            self.data[key] = [float(v) for v in value]
    def __setitem__(self, key, value):
        
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = float(value)
        else:
            if len(value) != self.cols:
                raise ValueError("Row length must match number of columns")
            self.data[key] = [float(v) for v in value]
    def transpose(self):
        transposed_data = [[self.data[i][j] for i in range(self.rows)] for j in range(self.cols)]
        return Matrix(transposed_data)
        
    def row(self, i):
        return Matrix([self.data[i]])

    def column(self, j):
        return Matrix([[self.data[i][j]] for i in range(self.rows)])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n0, n1, m0, m1):
        sub_data = [row[m0:m1] for row in self.data[n0:n1]]
        return Matrix(sub_data)
    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            if isinstance(i, slice) or isinstance(j, slice):
                sub_data = [row[j] for row in self.data[i]]
                return Matrix(sub_data)
            return self.data[i][j]
        elif isinstance(key, slice):
            return Matrix(self.data[key])
        return self.data[key]
   
    def scalarmul(self, c):
        new_data = [[value * c for value in row] for row in self.data]
        return Matrix(new_data)

    def add(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrices must have the same dimensions for addition")
        new_data = [[self.data[i][j] + N[i,j] for j in range(self.cols)] for i in range(self.rows)]
        return Matrix(new_data)

    def sub(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrices must have the same dimensions for subtraction")
        new_data = [[self.data[i][j] - N[i,j] for j in range(self.cols)] for i in range(self.rows)]
        return Matrix(new_data)

    def mat_mult(self, N):
        if self.cols != N.rows:
            raise ValueError("Matrix A columns must equal Matrix B rows for multiplication")
        new_data = [[sum(self.data[i][k] * N[k,j] for k in range(self.cols)) for j in range(N.cols)] for i in range(self.rows)]
        return Matrix(new_data)

    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrices must have the same dimensions for element-wise multiplication")
        new_data = [[self.data[i][j] * N[i,j] for j in range(self.cols)] for i in range(self.rows)]
        return Matrix(new_data)

    def __str__(self):
        formatted_rows = []
        for row in self.data:
            formatted_row = [f"{value:6.2f}" for value in row]
            formatted_rows.append(' '.join(formatted_row))
        return '\n'.join(formatted_rows)

    def __repr__(self):
        return self.__str__()
    
    def equals(self, N):
        if self.shape() != N.shape():
            return False
        return all(self.data[i][j] == N[i,j] for i in range(self.rows) for j in range(self.cols))
        
    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        elif isinstance(other, Matrix):
            return self.mat_mult(other)
        else:
            raise TypeError("Unsupported operand for *")

    def __rmul__(self, other):
        return self.__mul__(other)

    def __add__(self, other):
        if isinstance(other, Matrix):
            return self.add(other)
        else:
            raise TypeError("Unsupported operand for +")

    def __sub__(self, other):
        if isinstance(other, Matrix):
            return self.sub(other)
        else:
            raise TypeError("Unsupported operand for -")

    def __eq__(self, other):
        if isinstance(other, Matrix):
            return self.equals(other)
        return False

In [180]:
M1 = Matrix(2,3)
print(M1.shape())
print(M1.data)

M2 = Matrix([[1,2],[3,4]])
print(M2.shape())
print(M2.data)

(2, 3)
[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
(2, 2)
[[1.0, 2.0], [3.0, 4.0]]


In [181]:
M2 = Matrix([[1,2],[3,4]])

print(M2[0][1])
print(M2[0,1])

2.0
2.0


In [182]:
M = Matrix([[1,2],[3,4]])

M[0,1] = 10
print(M.data)  

M[1] = [7,8]
print(M.data)

[[1.0, 10.0], [3.0, 4.0]]
[[1.0, 10.0], [7.0, 8.0]]


In [183]:
M2 = Matrix([[1,2],[3,4]])

M2[0,1] = 10
print(M2.data)

M2[1] = [20,30]
print(M2.data)

[[1.0, 10.0], [3.0, 4.0]]
[[1.0, 10.0], [20.0, 30.0]]


In [184]:
M2 = Matrix([[1,2],[3,4]])
T = M2.transpose()
print(T.data)

[[1.0, 3.0], [2.0, 4.0]]


In [185]:
M2 = Matrix([[1,2],[3,4]])

print(M2.row(0).data)
print(M2.column(1).data)

[[1.0, 2.0]]
[[2.0], [4.0]]


In [186]:
print(M2.to_list())

[[1.0, 2.0], [3.0, 4.0]]


In [187]:
M2 = Matrix([[1,2,3],[4,5,6],[7,8,9]])
B = M2.block(0,2,1,3)
print(B.data)

[[2.0, 3.0], [5.0, 6.0]]


In [188]:
M = Matrix([[1,2,3],[4,5,6],[7,8,9]])
print(M[0:2,1:3].to_list())

[[2.0, 3.0], [5.0, 6.0]]


In [189]:
def constant(n, m, c):
    """Returns an n x m matrix filled with the float value c"""
    return Matrix([[float(c) for _ in range(m)] for _ in range(n)])
def zeros(n, m):
    """Returns an n x m matrix filled with 0.0"""
    return constant(n, m, 0.0)

def ones(n, m):
    """Returns an n x m matrix filled with 1.0"""
    return constant(n, m, 1.0)
    
def eye(n):
    """Returns an n x n identity matrix"""
    return Matrix([[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)])

In [190]:
A = zeros(2,3)
B = ones(3,3)
C = eye(3)

print(A.to_list())  
print(B.to_list())  
print(C.to_list()) 

[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
[[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]]
[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]


In [191]:
A = Matrix([[1,2],[3,4]])
B = Matrix([[5,6],[7,8]])

print(A.add(B).to_list())
print(A.sub(B).to_list())
print(A.scalarmul(2).to_list())
print(A.mat_mult(B).to_list())
print(A.element_mult(B).to_list())
print(A.equals(B))

[[6.0, 8.0], [10.0, 12.0]]
[[-4.0, -4.0], [-4.0, -4.0]]
[[2.0, 4.0], [6.0, 8.0]]
[[19.0, 22.0], [43.0, 50.0]]
[[5.0, 12.0], [21.0, 32.0]]
False


In [192]:
A = Matrix([[1,2],[3,4]])
B = Matrix([[5,6],[7,8]])

print(2 * A)
print(A * 2)
print(A + B)
print(A - B)
print(A * B)
print(A == B)

  2.00   4.00
  6.00   8.00
  2.00   4.00
  6.00   8.00
  6.00   8.00
 10.00  12.00
 -4.00  -4.00
 -4.00  -4.00
 19.00  22.00
 43.00  50.00
False


In [194]:
A = Matrix([[1, 2],[3, 4]])
B = Matrix([[5, 6],[7, 8]])
C = Matrix([[2, 0],[1, 2]])
I = eye(2)

print("Matrix A:\n", A)
print("Matrix B:\n", B)
print("Matrix C:\n", C)
print("Identity I:\n", I)
print("\n--- Demonstrating properties ---\n")

left = (A * B) * C
right = A * (B * C)
print("1) (AB)C = A(BC) ?")
print("Left side:\n", left)
print("Right side:\n", right)
print("Equal:", left == right, "\n")

left = A * (B + C)
right = (A * B) + (A * C)
print("2) A(B+C) = AB + AC ?")
print("Left side:\n", left)
print("Right side:\n", right)
print("Equal:", left == right, "\n")

AB = A * B
BA = B * A
print("3) AB vs BA")
print("AB:\n", AB)
print("BA:\n", BA)
print("Equal:", AB == BA, "\n")

AI = A * I
print("4) AI = A ?")
print("AI:\n", AI)
print("A:\n", A)
print("Equal:", AI == A)

Matrix A:
   1.00   2.00
  3.00   4.00
Matrix B:
   5.00   6.00
  7.00   8.00
Matrix C:
   2.00   0.00
  1.00   2.00
Identity I:
   1.00   0.00
  0.00   1.00

--- Demonstrating properties ---

1) (AB)C = A(BC) ?
Left side:
  60.00  44.00
136.00 100.00
Right side:
  60.00  44.00
136.00 100.00
Equal: True 

2) A(B+C) = AB + AC ?
Left side:
  23.00  26.00
 53.00  58.00
Right side:
  23.00  26.00
 53.00  58.00
Equal: True 

3) AB vs BA
AB:
  19.00  22.00
 43.00  50.00
BA:
  23.00  34.00
 31.00  46.00
Equal: False 

4) AI = A ?
AI:
   1.00   2.00
  3.00   4.00
A:
   1.00   2.00
  3.00   4.00
Equal: True
